In [2]:
import pandas as pd
import numpy as np
from pybaseball import statcast, cache
from tqdm.notebook import tqdm
import os
import warnings
import shutil
import requests

# 1. Environment Setup & Cache Cleaning
# Clearing physical cache to prevent 'BadZipFile' errors
cache_dir = os.path.expanduser('~/.pybaseball/cache')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("✅ Cleaned corrupted cache directory.")

# pybaseball 자체 캐시 기능을 꺼서 쓰레기 데이터 로드 원천 차단
cache.disable()
tqdm.pandas()
warnings.filterwarnings('ignore')

# Directory Setup
base_dir = os.path.dirname(os.getcwd())
output_dir = os.path.join(base_dir, 'data', 'aging_project')
os.makedirs(output_dir, exist_ok=True)

YEARS = [2022, 2023, 2024]

# ---------------------------------------------------------
# 2. Metric Aggregation Logic
# ---------------------------------------------------------

def calculate_hitter_metrics(group, year):
    """Aggregates Statcast hitting data into player-season metrics."""
    pa = group.groupby(['game_date', 'at_bat_number']).size().shape[0]
    
    # AVG (Hits / At-Bats)
    hits = len(group[group['events'].isin(['single', 'double', 'triple', 'home_run'])])
    ab_events = group[~group['events'].isin(['walk', 'hit_by_pitch', 'sac_fly', 'sac_bunt', None])]
    denominator = len(ab_events)
    avg = hits / denominator if denominator > 0 else 0
    
    # Chase% (Swings at pitches outside the strike zone)
    out_of_zone = group[group['zone'] > 10]
    swings_out = out_of_zone[out_of_zone['description'].str.contains('swing|foul|hit_into_play', na=False)]
    chase_pct = len(swings_out) / len(out_of_zone) if len(out_of_zone) > 0 else 0
    
    return pd.Series({
        'pa': pa,
        'avg': avg,
        'chase_pct': chase_pct,
        'avg_ev': group['launch_speed'].mean(),
        'season': year
    })

def calculate_pitcher_metrics(group, year):
    """Aggregates Statcast pitching data into player-season metrics."""
    pitches = len(group)
    # Focus on Four-Seam Fastballs (FF) for aging analysis
    ff_group = group[group['pitch_type'] == 'FF']
    
    fb_velocity = ff_group['release_speed'].mean() if not ff_group.empty else np.nan
    spin_rate = ff_group['release_spin_rate'].mean() if not ff_group.empty else np.nan
    
    # Chase% for pitchers
    out_of_zone = group[group['zone'] > 10]
    swings_out = out_of_zone[out_of_zone['description'].str.contains('swing|foul|hit_into_play', na=False)]
    chase_pct = len(swings_out) / len(out_of_zone) if len(out_of_zone) > 0 else 0
    
    return pd.Series({
        'pitches': pitches,
        'fb_velocity': fb_velocity,
        'spin_rate': spin_rate,
        'chase_pct': chase_pct,
        'season': year
    })

def process_statcast_aging(year):
    """Orchestrates data collection for a single season."""
    print(f"\n🚀 Fetching {year} Season Data...")
    start_dt = f"{year}-04-01"
    end_dt = f"{year}-10-01"
    
    # Statcast API call
    raw = statcast(start_dt=start_dt, end_dt=end_dt)
    raw.columns = raw.columns.str.lower()
    
    # Progress bars for aggregation
    tqdm.pandas(desc=f" └─ Hitters ({year})", leave=False, position=1)
    hitter_stats = raw.groupby('batter').progress_apply(calculate_hitter_metrics, year=year, include_groups=False).reset_index()

    tqdm.pandas(desc=f" └─ Pitchers ({year})", leave=False, position=1)
    pitcher_stats = raw.groupby('pitcher').progress_apply(calculate_pitcher_metrics, year=year, include_groups=False).reset_index()

    return hitter_stats[hitter_stats['pa'] >= 200], pitcher_stats[pitcher_stats['pitches'] >= 500]

# ---------------------------------------------------------
# 3. Execution Main Loop
# ---------------------------------------------------------

all_hitters_list = []
all_pitchers_list = []

main_pbar = tqdm(YEARS, desc="Total Project Progress", position=0)

for y in main_pbar:
    main_pbar.set_description(f"Current Season: {y}")
    h_data, p_data = process_statcast_aging(y)
    all_hitters_list.append(h_data)
    all_pitchers_list.append(p_data)

# Merging yearly results
print("\n🔗 Starting metadata mapping via MLB Stats API...")
hitter_master_raw = pd.concat(all_hitters_list, ignore_index=True)
pitcher_master_raw = pd.concat(all_pitchers_list, ignore_index=True)

# ---------------------------------------------------------
# 4. Final Metadata Mapping (MLB API Bypass)
# ---------------------------------------------------------

def finalize_data_collection_api(df, id_col):
    """
    Bypasses pybaseball completely and fetches player birth dates 
    directly from the official MLB Stats API.
    """
    print(f"  > Processing {id_col} metadata...")
    
    df[id_col] = df[id_col].astype(int)
    unique_ids = df[id_col].unique().tolist()
    
    # Chunking IDs to avoid overly long URLs
    chunk_size = 300
    player_data = []
    
    for i in range(0, len(unique_ids), chunk_size):
        chunk = unique_ids[i:i + chunk_size]
        id_str = ",".join(map(str, chunk))
        
        # Official MLB API Endpoint
        url = f"https://statsapi.mlb.com/api/v1/people?personIds={id_str}"
        
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                for person in data.get('people', []):
                    player_data.append({
                        'key_mlbam': person.get('id'),
                        'full_name': person.get('fullName'),
                        'birth_date': person.get('birthDate')
                    })
        except Exception as e:
            print(f"  ⚠️ API request failed for chunk: {e}")
            
    # API 결과를 데이터프레임으로 변환
    ppl_df = pd.DataFrame(player_data)
    
    if ppl_df.empty:
        print("  ❌ API returned no data. Check internet connection.")
        return pd.DataFrame()
        
    # 생년월일 추출
    ppl_df['birth_year_clean'] = pd.to_datetime(ppl_df['birth_date'], errors='coerce').dt.year
    
    # 조인 및 나이 계산
    merged = pd.merge(df, ppl_df, left_on=id_col, right_on='key_mlbam', how='inner')
    merged['age'] = merged['season'] - merged['birth_year_clean']
    
    return merged.drop(columns=['key_mlbam', 'birth_date', id_col]).dropna(subset=['age'])
    
# Final Mapping
final_hitters = finalize_data_collection_api(hitter_master_raw, 'batter')
final_pitchers = finalize_data_collection_api(pitcher_master_raw, 'pitcher')

# Save Results
final_hitters.to_csv(os.path.join(output_dir, 'aging_hitters_master.csv'), index=False)
final_pitchers.to_csv(os.path.join(output_dir, 'aging_pitchers_master.csv'), index=False)

print("\n" + "="*50)
print(f"✅ Data Collection Complete!")
print(f"📊 Final Hitter Sample: {len(final_hitters)}")
print(f"📊 Final Pitcher Sample: {len(final_pitchers)}")
print(f"📂 Location: {output_dir}")
print("="*50)

Total Project Progress:   0%|          | 0/3 [00:00<?, ?it/s]


🚀 Fetching 2022 Season Data...
This is a large query, it may take a moment to complete



100%|██████████| 184/184 [01:00<00:00,  3.05it/s]


 └─ Hitters (2022):   0%|          | 0/1183 [00:00<?, ?it/s]

 └─ Pitchers (2022):   0%|          | 0/1004 [00:00<?, ?it/s]

...


🚀 Fetching 2024 Season Data...
This is a large query, it may take a moment to complete



100%|██████████| 184/184 [01:07<00:00,  2.73it/s]


 └─ Hitters (2024):   0%|          | 0/651 [00:00<?, ?it/s]

 └─ Pitchers (2024):   0%|          | 0/853 [00:00<?, ?it/s]


🔗 Starting metadata mapping via MLB Stats API...
  > Processing batter metadata...
  > Processing pitcher metadata...

✅ Data Collection Complete!
📊 Final Hitter Sample: 1086
📊 Final Pitcher Sample: 1420
📂 Location: /Users/heounnyeong/Desktop/Personal_3/data/aging_project
